# Count-controlled Binding over training

Reads **existing per-question predictions**; no model loading or training. Validates exact question IDs and labels against the frozen `id_dev`/`ood_dev` JSONL files. Reports the count-free subset and all-Pair comparison, five-seed paired changes, and mean trajectories. `strict=False` allows only the locally present studies; all missing sources are logged.

A count-free question has exactly one target-Color object and exactly one target-Shape object. This eliminates simple multiplicity cues but does **not** eliminate size, placement, scene-composition, or OOD-label shortcuts. Balanced accuracy is primary because the subset has an imbalanced Yes/No prior.

In [1]:
from pathlib import Path
import sys
PROJECT_ROOT = Path().cwd().resolve().parent.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0,str(PROJECT_ROOT))
from src.medium_visual_world.diagnostic_followup import *
OUTPUT = PROJECT_ROOT / "outputs/medium_world/experiments/diagnostic_followup"
OUTPUT.mkdir(parents=True,exist_ok=True)

### Run using saved `.npz` prediction traces; no retraining

In [2]:
FAMILIES = ("phase_aware_lr", "mixed_pair_control/five_seeds", "scaffolded_spiral")
curves,integrity,differences = save_curve_analysis(PROJECT_ROOT,OUTPUT/"count_free",FAMILIES,strict=False)
print("Completed run-traces:",len(integrity)//2)
print("Count-free ID support per run:",sorted(integrity[integrity.split=="id_dev"].count_free.unique()))
print("Count-free OOD support per run:",sorted(integrity[integrity.split=="ood_dev"].count_free.unique()))
print("Missing source count:",len(__import__('json').loads((OUTPUT/"count_free/missing_sources.json").read_text())))
display(curves[(curves.split=="id_dev")&(curves.subset=="count_free")&(curves.step.isin((5400,7200,9000)))].groupby(["study","policy","schedule","step"])[["balanced_accuracy","accuracy","tpr","tnr"]].mean().round(3))
display(differences.round(4))

Completed run-traces: 145
Count-free ID support per run: [np.int64(109)]
Count-free OOD support per run: [np.int64(154)]
Missing source count: 0


balanced_accuracy  \
study                         policy     schedule          step                      
mixed_pair_control/five_seeds historical mixed             5400              0.539   
                                                           7200              0.592   
                                                           9000              0.618   
                                         mixed_pair_focus  5400              0.528   
                                                           7200              0.639   
...                                                                            ...   
scaffolded_spiral             historical spiral_late_focus 7200              0.624   
                                                           9000              0.591   
                                         staged_rehearsal  5400              0.526   
                                                           7200              0.642   
                                                           9000              0.624   

                                                                 accuracy  \
study                         policy     schedule          step             
mixed_pair_control/five_seeds historical mixed             5400     0.512   
                                                           7200     0.549   
                                                           9000     0.567   
                                         mixed_pair_focus  5400     0.483   
                                                           7200     0.585   
...                                                                   ...   
scaffolded_spiral             historical spiral_late_focus 7200     0.591   
                                                           9000     0.574   
                                         staged_rehearsal  5400     0.534   
                                                           7200     0.609   
                                                           9000     0.598   

                                                                   tpr    tnr  
study                         policy     schedule          step                
mixed_pair_control/five_seeds historical mixed             5400  0.612  0.467  
                                                           7200  0.706  0.477  
                                                           9000  0.753  0.483  
                                         mixed_pair_focus  5400  0.647  0.408  
                                                           7200  0.782  0.496  
...                                                                ...    ...  
scaffolded_spiral             historical spiral_late_focus 7200  0.712  0.536  
                                                           9000  0.635  0.547  
                                         staged_rehearsal  5400  0.506  0.547  
                                                           7200  0.729  0.555  
                                                           9000  0.694  0.555  

[87 rows x 4 columns]

,study,policy,schedule,start,end,n_seeds,mean_change,ci95_low,ci95_high,positive_seeds
0,mixed_pair_control/five_seeds,historical,mixed,5400,7200,5,0.0524,-0.0186,0.1234,5
1,mixed_pair_control/five_seeds,historical,mixed,7200,9000,5,0.0262,0.0113,0.0411,5
2,mixed_pair_control/five_seeds,historical,mixed_pair_focus,5400,7200,5,0.1116,0.0983,0.1250,5
3,mixed_pair_control/five_seeds,historical,mixed_pair_focus,7200,9000,5,-0.0113,-0.0371,0.0146,1
4,mixed_pair_control/five_seeds,historical,prereq_mixed_pair_focus,5400,7200,5,0.2007,0.0760,0.3255,5
5,mixed_pair_control/five_seeds,historical,prereq_mixed_pair_focus,7200,9000,5,-0.0468,-0.1430,0.0495,1
6,mixed_pair_control/five_seeds,historical,staged_pair_focus,5400,7200,5,0.1628,0.1274,0.1982,5
7,mixed_pair_control/five_seeds,historical,staged_pair_focus,7200,9000,5,-0.0249,-0.0887,0.0390,2
8,phase_aware_lr,constant,mixed,5400,7200,5,0.0521,-0.0192,0.1235,5
9,phase_aware_lr,constant,mixed,7200,9000,5,0.0347,0.0097,0.0598,5


### Inspect policy-by-curriculum effects, not just the final 9,000 updates

In [3]:
valid=curves[(curves.split=="id_dev")&(curves.subset=="count_free")]
last=valid[valid.step==9000].groupby(["study","policy","schedule"])[["balanced_accuracy","tpr","tnr"]].agg(["mean","std"]).round(3)
display(last)
print("See outputs/medium_world/experiments/diagnostic_followup/count_free for CSVs and figures")

balanced_accuracy  \
                                                                                  mean   
study                         policy         schedule                                    
mixed_pair_control/five_seeds historical     mixed                               0.618   
                                             mixed_pair_focus                    0.628   
                                             prereq_mixed_pair_focus             0.636   
                                             staged_pair_focus                   0.622   
phase_aware_lr                constant       mixed                               0.623   
                                             prereq_mixed_pair_focus             0.636   
                                             scaffolded_spiral                   0.597   
                                             staged_rehearsal                    0.630   
                              global_cosine  mixed                               0.509   
                                             prereq_mixed_pair_focus             0.523   
                                             scaffolded_spiral                   0.521   
                                             staged_rehearsal                    0.527   
                              global_plateau mixed                               0.540   
                                             prereq_mixed_pair_focus             0.453   
                                             scaffolded_spiral                   0.478   
                                             staged_rehearsal                    0.460   
                              reset_cosine   mixed                               0.512   
                                             prereq_mixed_pair_focus             0.535   
                                             scaffolded_spiral                   0.543   
                                             staged_rehearsal                    0.544   
                              reset_plateau  mixed                               0.531   
                                             prereq_mixed_pair_focus             0.547   
                                             scaffolded_spiral                   0.579   
                                             staged_rehearsal                    0.580   
scaffolded_spiral             historical     mixed                               0.625   
                                             prereq_mixed_pair_focus             0.635   
                                             scaffolded_spiral                   0.594   
                                             spiral_late_focus                   0.591   
                                             staged_rehearsal                    0.624   

                                                                             \
                                                                        std   
study                         policy         schedule                         
mixed_pair_control/five_seeds historical     mixed                    0.031   
                                             mixed_pair_focus         0.040   
                                             prereq_mixed_pair_focus  0.043   
                                             staged_pair_focus        0.057   
phase_aware_lr                constant       mixed                    0.038   
                                             prereq_mixed_pair_focus  0.038   
                                             scaffolded_spiral        0.057   
                                             staged_rehearsal         0.021   
                              global_cosine  mixed                    0.048   
                                             prereq_mixed_pair_focus  0.054   
                                             scaffolded_spiral        0.061   
                                             staged_rehearsal         0.052   
   

See outputs/medium_world/experiments/diagnostic_followup/count_free for CSVs and figures
